**Import libraries and load dataset**

In [30]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = "academic_success_dataset.csv"
df = pd.read_csv(DATA_PATH)

df.columns = df.columns.str.strip()
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (4424, 25)
   Marital status  Application mode  Application order  Course  \
0             1.0              17.0                5.0   171.0   
1             1.0              15.0                1.0  9254.0   
2             1.0               NaN                5.0  9070.0   
3             1.0              17.0                2.0  9773.0   
4             2.0              39.0                1.0  8014.0   

   Daytime/evening attendance  Previous qualification  \
0                         1.0                     1.0   
1                         1.0                     1.0   
2                         1.0                     1.0   
3                         1.0                     1.0   
4                         0.0                     1.0   

   Previous qualification (grade)  Nacionality  Mother's qualification  \
0                           122.0          1.0                    19.0   
1                           160.0          1.0                     1.0   
2           

**Check target and missing values**

In [31]:
print("Target values:")
print(df["Target"].value_counts(dropna=False))

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

Target values:
Target
Graduate    1979
Dropout     1273
Enrolled     719
NaN          453
Name: count, dtype: int64

Missing values:
Application mode                  483
Previous qualification (grade)    472
Course                            465
Admission grade                   462
GDP                               456
Target                            453
Debtor                            452
Scholarship holder                450
Father's qualification            450
Educational special needs         448
Nacionality                       446
Age at enrollment                 444
Displaced                         442
Daytime/evening attendance        440
Gender                            437
Mother's occupation               436
Previous qualification            434
Unemployment rate                 429
International                     428
Application order                 426
Tuition fees up to date           426
Father's occupation               425
Marital status                 

**Prepare features and target**

In [32]:
df_model = df.dropna(subset=["Target"]).copy()

X = df_model.drop(columns=["Target"])
y = df_model["Target"]

categorical_cols = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification", "Nacionality",
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder", "International"
]
categorical_cols = [c for c in categorical_cols if c in X.columns]

numerical_cols = [c for c in X.columns if c not in categorical_cols]

print("Rows used for training:", len(df_model))
print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", numerical_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Rows used for training: 3971
Categorical columns: 18
Numerical columns: ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Unemployment rate', 'Inflation rate', 'GDP']


**Data preprocessing**

In [33]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", onehot)
        ]), categorical_cols),

        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numerical_cols)
    ]
)

**Train models**

In [34]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Linear SVM": LinearSVC(class_weight="balanced", max_iter=5000, random_state=42),
    "SVM RBF": SVC(kernel="rbf", class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced", n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced")
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(y_test, y_pred, average="macro")
    })

    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
results_df

,Model,Accuracy,Macro F1
1,Linear SVM,0.617610,0.554524
2,SVM RBF,0.562264,0.538684
0,Logistic Regression,0.552201,0.523285
4,Gradient Boosting,0.616352,0.485916
3,Random Forest,0.610063,0.470593
5,Decision Tree,0.513208,0.457922


**Evaluate best model**

In [35]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

y_pred = best_model.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
labels = ["Dropout", "Enrolled", "Graduate"]
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f"Actual {x}" for x in labels],
    columns=[f"Predicted {x}" for x in labels]
))

Best model: Linear SVM

Classification Report:
              precision    recall  f1-score   support

     Dropout       0.67      0.55      0.60       255
    Enrolled       0.33      0.34      0.34       144
    Graduate       0.69      0.76      0.72       396

    accuracy                           0.62       795
   macro avg       0.56      0.55      0.55       795
weighted avg       0.62      0.62      0.61       795

Confusion Matrix:
                 Predicted Dropout  Predicted Enrolled  Predicted Graduate
Actual Dropout                 140                  47                  68
Actual Enrolled                 26                  49                  69
Actual Graduate                 43                  51                 302
